<a href="https://colab.research.google.com/github/DangLeUyen/Reinforcement-Learning/blob/main/U2_frozenlake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Q-Learning algorithm from scratch

1. Install dependencies and create a virtual display, including `gymnasium`, `pygame`, and `numpy`
2.

In [7]:
pip install gymnasium pygame huggingface_hub imageio imageio_ffmpeg pyyaml pyglet

In [8]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,357 kB]
Get:14 http:/

In [ ]:
# force the runtime to crash
import os

os.kill(os.getpid(), 9)


In [1]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()


In [6]:
# Import the packages
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle
from tqdm.notebook import tqdm

### 2. Create and understand FrozenLake environment
(( https://gymnasium.farama.org/environments/toy_text/frozen_lake/ )

We train Q-Learning agent to navigate from the starting state (S) to the goal state (G) by walking only on frozen tiles (F) and avoid holes (H).

There are two sizes of environment:
- `map_name`="4x4": a 4x4 grid version
- `map_name`="8x8": a 8x8 grid version

The environment has two modes:
- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
- `is_slippery=True`: The agent may not always move in the intended direction due to the slippery nature of the frozen lake (stochastic).

**Keep it simple with the 4x4 map and non-slippery**


In [8]:
# Create the FrozenLake-v1 environment using 4x4 map and non-slippery version and render_mode="rgb_array"
env = gym.make("FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array")


In [9]:
# See the environment
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample())  # Get a random observation


_____OBSERVATION SPACE_____ 

Observation Space Discrete(16)
Sample observation 13


In [10]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())  # Take a random action



 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 0


### 3. Create and Initialize the Q-table

In [12]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")


There are  16  possible states
There are  4  possible actions


In [13]:
# Create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable


In [14]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

### 4. Define the greedy policy